In [ ]:
import numpy as np
import skops.io as sio
import os
from matplotlib import pyplot as plt
import pandas as pd
import seaborn as sns

if os.getcwd().endswith("notebooks"):
    os.chdir("..")

In [ ]:
from src.model import CircularTransformer, greenup_activity, sos_swq_tmin_predictor

### Prepare climate data

Here we need a version at annual timestep and a version at daily timestep. Use `BNU-ESM` for now.

In [ ]:
maca = pd.read_parquet("phenocam_site_in_usa_climate_maca.parquet")
# Drop leap years
maca = maca[maca.index.get_level_values("time").dayofyear <= 365]

# Years not needed in column name because time already indexes
maca = (
    pd.melt(maca, ignore_index=False)
        .dropna()
        .assign(variable = lambda x: x.variable.str.extract(r"^([^_]+_[^_]+)"))
        .pivot(columns="variable", values="value")
)

meta = pd.read_parquet("phenocam_site_in_usa.parquet")

In [ ]:
maca.head()

In [ ]:
ESM = "IPSL-CM5A-MR"

maca_daily = (
    maca
    .filter(like=ESM)
    .groupby(["site", maca.index.get_level_values("time").year])
    .apply(lambda g: pd.Series({v: g[v].values for v in g.columns}))
)

maca_daily["veg_type"] = meta["primary_veg_type"].loc[maca_daily.index.get_level_values("site")].to_numpy()
maca_daily["kop_type"] = meta["koeppen_geiger"].loc[maca_daily.index.get_level_values("site")].str[0:2].to_numpy()
maca_daily = maca_daily.dropna(subset=["veg_type", "kop_type"])

maca_monthly = (
    maca
    .filter(like=ESM)
    .groupby([
        "site", 
        maca.index.get_level_values("time").year.rename("year"), 
        maca.index.get_level_values("time").month.rename("month")
    ])
    .agg({
        ESM+"_pr": "sum",
        ESM+"_tasmax": lambda x: x.mean() - 273.15 # as deg C
    })
    .rename(columns={
        ESM+"_pr": "prcp",
        ESM+"_tasmax": "tmax"
    })
    .reset_index(level="month")
    .assign(month = lambda x: "month" + x.month.astype(str))
    .pivot(columns="month")
    .dropna()
)

maca_monthly.columns = ['_'.join(col).strip() for col in maca_monthly.columns.values]
maca_monthly["veg_type"] = meta["primary_veg_type"].loc[maca_monthly.index.get_level_values("site")].to_numpy()
maca_monthly.head()

### Load models

In [ ]:
untrusted_types = sio.get_untrusted_types(file="phenocam_monthly_plsr.skops")
plsr_model = sio.load("phenocam_monthly_plsr.skops", trusted=untrusted_types)

In [ ]:
forcing_model_veg = pd.read_parquet("phenocam_forcing_model_parameters_veg_type.parquet").T
forcing_model_kop = pd.read_parquet("phenocam_forcing_model_parameters_kop_type.parquet").T

param_table_veg = forcing_model_veg.loc[maca_daily["veg_type"].values]
param_table_kop = forcing_model_kop.loc[maca_daily["kop_type"].values]

### Run predictions

In [ ]:
# Forcing model
temp_arrays = np.stack(maca_daily[f"{ESM}_tasmax"], axis=1).T
prcp_arrays = np.stack(maca_daily[f"{ESM}_pr"], axis=1).T

est_greenup_doy_veg = sos_swq_tmin_predictor(
    temp_arrays, prcp_arrays,
    param_table_veg[["b"]].to_numpy(),
    param_table_veg[["c"]].to_numpy(),
    param_table_veg[["Preq"]].to_numpy(),
    param_table_veg[["Tthres"]].to_numpy(),
    param_table_veg[["Tb"]].to_numpy(),
    param_table_veg[["Fcrit"]].to_numpy()
) + 1

est_greenup_doy_kop = sos_swq_tmin_predictor(
    temp_arrays, prcp_arrays,
    param_table_kop[["b"]].to_numpy(),
    param_table_kop[["c"]].to_numpy(),
    param_table_kop[["Preq"]].to_numpy(),
    param_table_kop[["Tthres"]].to_numpy(),
    param_table_kop[["Tb"]].to_numpy(),
    param_table_kop[["Fcrit"]].to_numpy()
) + 1

maca_daily["forcing_model_veg_greenup_doy"] = est_greenup_doy_veg
maca_daily["forcing_model_kop_greenup_doy"] = est_greenup_doy_kop

In [ ]:
# PLSR model
maca_monthly["model_greenup_doy"] = plsr_model.predict(maca_monthly[plsr_model.feature_names_in_])

### Distribution of greenup DOY by veg type

In [ ]:
grid = sns.FacetGrid(
    maca_monthly.assign(period = lambda x: np.where(x.index.get_level_values("year") < 2066, "2006 - 2025", "2066 - 2085")),
    col="veg_type",
    col_wrap=3,
    hue="period",
    xlim=(0, 365)
)

grid.map(sns.kdeplot, 'model_greenup_doy', fill=True, alpha=0.5)
grid.add_legend(title="")
grid.set_titles(template="{col_name}")
grid.set_xlabels("")
grid.set_ylabels("")
grid.fig.subplots_adjust(top=0.85) 
grid.fig.suptitle('PLSR Model', fontsize=16)

In [ ]:
grid = sns.FacetGrid(
    maca_daily.assign(period = lambda x: np.where(x.index.get_level_values("time") < 2066, "2006 - 2025", "2066 - 2085")),
    col="veg_type",
    col_wrap=3,
    hue="period",
    xlim=(0, 365)
)

grid.map(sns.kdeplot, 'forcing_model_veg_greenup_doy', fill=True, alpha=0.5)
grid.add_legend(title="")
grid.set_titles(template="{col_name}")
grid.set_xlabels("")
grid.set_ylabels("")
grid.fig.subplots_adjust(top=0.85) 
grid.fig.suptitle('Forcing Model (by veg type)', fontsize=16)

In [ ]:
grid = sns.FacetGrid(
    maca_daily.assign(period = lambda x: np.where(x.index.get_level_values("time") < 2066, "2006 - 2025", "2066 - 2085")),
    col="veg_type",
    col_wrap=3,
    hue="period",
    xlim=(0, 365)
)

grid.map(sns.kdeplot, 'forcing_model_kop_greenup_doy', fill=True, alpha=0.5)
grid.add_legend(title="")
grid.set_titles(template="{col_name}")
grid.set_xlabels("")
grid.set_ylabels("")
grid.fig.subplots_adjust(top=0.85) 
grid.fig.suptitle('Forcing Model (by climate type)', fontsize=16)

Plot site differences by model

In [ ]:
maca_daily_site_mean = (
    maca_daily
    .assign(period = lambda x: np.where(x.index.get_level_values("time") < 2066, "2006 - 2025", "2066 - 2085"))
    .groupby(["period", "site"])
    [["forcing_model_veg_greenup_doy", "forcing_model_kop_greenup_doy"]]
    .mean()
)

forcing_model_site_delta = maca_daily_site_mean.loc["2066 - 2085"] - maca_daily_site_mean.loc["2006 - 2025"]

In [ ]:
maca_monthly_site_mean = (
    maca_monthly
    .assign(period = lambda x: np.where(x.index.get_level_values("year") < 2066, "2006 - 2025", "2066 - 2085"))
    .rename(columns=dict(model_greenup_doy="plsr_model_greenup_doy"))
    .groupby(["period", "site"])
    [["plsr_model_greenup_doy"]]
    .mean()
)

plsr_model_site_delta = maca_monthly_site_mean.loc["2066 - 2085"] - maca_monthly_site_mean.loc["2006 - 2025"]

In [ ]:
all_model_site_delta = (
    plsr_model_site_delta
    .join(forcing_model_site_delta)
    .assign(veg_type = meta["primary_veg_type"].loc[plsr_model_site_delta.index.get_level_values("site")].to_numpy())
)

all_model_site_delta_melt = pd.melt(
    all_model_site_delta,
    value_vars=[
        "plsr_model_greenup_doy",
        "forcing_model_veg_greenup_doy",
        "forcing_model_kop_greenup_doy"
    ],
    id_vars=["veg_type"],
    ignore_index=False
).assign(
    variable=lambda x: x.variable.map({
        "plsr_model_greenup_doy": "PLSR, monthly",
        "forcing_model_veg_greenup_doy": "Forcing, by veg type",
        "forcing_model_kop_greenup_doy": "Forcing, by climate"
    })
)

In [ ]:
grid = sns.FacetGrid(
    all_model_site_delta_melt,
    col="veg_type",
    col_wrap=3,
    hue="variable",
    xlim=(-50, 50),
    despine=False
)

grid.refline(x=0, color="black", linestyle="--")
grid.map(sns.kdeplot, 'value', fill=True, alpha=0.5)
grid.add_legend(title="")
grid.set_titles(template="{col_name}")
grid.set_xlabels("")
grid.set_ylabels("")
grid.fig.supxlabel("Average change in greenup DOY, 2066-2085 vs. 2006-2025")

plt.savefig("figures/greenup_doy_change_by_model_veg.png")
plt.show()